# YAAPT và AMDF: tái lập từ WAV

F0 là tần số cơ bản. V/UV/SIL là hữu thanh/vô thanh/khoảng lặng. YAAPT dùng ứng viên từ phổ và tương quan chuẩn hóa (tìm chu kỳ), rồi quy hoạch động chọn chuỗi qua thời gian. AMDF dùng chênh lệch biên độ trung bình theo độ trễ. Notebook đo lại H24 AMDF và toàn bộ H40 đã đăng ký; không mở grid mới. Average MAPE trung bình lỗi của mean/std(ddof0)/count; mục tiêu mỗi file≤2%. LAB không có F0 chuẩn từng khung.


In [ ]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
REPO=Path('C:\\Users\\LAPTOP T&T\\VIOLETTA\\Documents\\ChatGPT\\XLTN\\XLTN-BT2')
HERE=REPO/'research_workbench_2026_10_07'
sys.path.insert(0,str(HERE))
import amdf_dual_window as dual
import yaapt_reference as reference
core,audit=dual.core,dual.audit
items=core.load_training()
by_name={x['file']:x for x in items}
config=json.loads((core.RESULTS/'frozen_config.json').read_text())['models']['AMDF_energy']['config']
families={'H40':reference}
receipts={f:json.loads((HERE/f'results/{f}_experiment.json').read_text()) for f in families}
for receipt in receipts.values():
    for relative,expected in receipt['code_sha256'].items():
        assert audit.digest(REPO/relative)==expected,relative
    for file,expected in receipt['data_sha256'].items():
        assert audit.digest(core.TRAIN/file)==expected,file
native_proof=reference.native_api.metadata()
yaapt_proof=reference.yaapt_api.metadata()
print('Local Python:',sys.executable,'; Praat',native_proof['version_stdout'],'; YAAPT port',yaapt_proof['package_version'])
display(pd.DataFrame([{'file':x['file'],'fs':x['fs'],**x['stats'],'segment_V_frames':int((x['labels']=='v').sum())} for x in items]))


## Tính lại mọi cấu hình fixed

H24 AMDF fit bằng ba file khác. YAAPT/Praat không fit theo nhãn. SourceYAAPT1.0.12.2/pin5c6c9bc, frame_length25/35/45 và tda35, hop10ms/range70–400. Spectral tracking dùng2×frame_size. Float64 normalized PCM giữ; lấy samp_values UV0, không nội suy lấpUV. Native centers ghép gần nhất vào lưới chấm25/10 trong5ms+mộtmẫu, hòa chọn sớm. Causal FIR/padding/vendorDP giữ. Không resample/clip/noise/offset theoLAB.


In [ ]:
registries={f:json.loads((HERE/f'{f}_REGISTRY.json').read_text())['options'] for f in families}
features={}
native_calls={}
rows,contour_rows,fit_logs=[],[],[]
columns=['F0mean','F0std','F0num','F0mean_mape','F0std_mape','F0num_mape','average_mape',
         'macro_f1','recall_v','recall_uv','balanced_accuracy','TP','FN','FP','TN','false_voiced_sil']

def feature(module,option,item):
    key=(module.__name__,module.feature_key(option),item['file'])
    if key not in features:
        _,audio=core.load_audio(core.TRAIN/item['file'])
        features[key]=module.extract(item,audio,option)
        if 'native_call' in features[key]:
            native_calls['|'.join(key)]=features[key]['native_call']
    return features[key]

def measure(family,module,option,held,split):
    pool=[x for x in items if x['file']!=held['file']]
    native=feature(module,option,held)
    training=[feature(module,option,x) for x in pool]
    pred,f0,fit,_=module.infer(native,training,option,config)
    if module is dual:
        fit=dict(fit,requires_fit=True,actual_fit_files=sorted(x['file'] for x in pool))
    pred,f0,support=module.project(native,held,pred,f0,option['hop_ms'])
    measured=core.score_file(held,pred,f0)
    saved=pd.read_csv(HERE/f'results/{family}_fixed_lofo.csv') if split=='fixed' else pd.read_csv(HERE/f'results/{family}_metrics.csv')
    saved=saved[(saved.option_id==option['id'])&(saved.file==held['file'])]
    if split=='nested':
        saved=saved[(saved.split=='nested')&(saved.model=='candidate')]
    assert len(saved)==1
    assert np.allclose([measured[k] for k in columns],saved.iloc[0][columns].astype(float),atol=1e-8),(family,option['id'],held['file'])
    method=option['id'] if split=='fixed' else family+'_nested'
    rows.append({'evaluation':split,'method':method,'family':family,'option_id':option['id'],**measured})
    fit_logs.append({'evaluation':split,'method':method,'held_file':held['file'],'fit_files':sorted(x['file'] for x in pool),'fitted':fit})
    contour_rows.extend({'evaluation':split,'method':method,'family':family,'file':held['file'],
                        'time_s':float(t),'label':str(lab),'pred_voiced':bool(p),'f0_hz':float(f),'support':bool(s)}
                       for t,lab,p,f,s in zip(held['times'],held['labels'],pred,f0,support))

for held in items:
    measure('H24',dual,dual.registry('H24')[1],held,'fixed')
for family,module in families.items():
    for option in registries[family]:
        for held in items:
            measure(family,module,option,held,'fixed')
display(pd.DataFrame(rows)[['family','option_id','file','average_mape','F0mean_mape','F0std_mape','F0num_mape','false_voiced_sil']])
print('PASS20 fixed rows được tính từ WAV, khớp số liệu đã lưu 1e-8.')


## Chấm file được giữ riêng

Nested giữ một outer file ngoài mọi lựa chọn; ba file còn lại chọn option theo minimax lỗi từng file rồi mean/ID. Notebook dùng đúng choice đã lưu, không chọn best theo held file. Lịch sử đã xem bốn file khiến nested vẫn exploratory, chưa xác nhận trên corpus mới.


In [ ]:
for family,module in families.items():
    for selection in receipts[family]['selections']:
        held=selection['outer_held']
        if held=='final':
            continue
        assert held not in selection['selection_files'] and set(selection['selection_files'])==set(by_name)-{held}
        measure(family,module,selection['option'],by_name[held],'nested')
table=pd.DataFrame(rows)
contour_table=pd.DataFrame(contour_rows)
assert len(table)==24
table.to_csv(HERE/'results/yaapt_notebook_per_file.csv',index=False)
contour_table.to_csv(HERE/'results/yaapt_notebook_contours.csv',index=False)
status=table.query("evaluation=='nested'").groupby('family').average_mape.agg(mean='mean',worst='max',all_files_le_2=lambda x:bool((x<=2).all())).reset_index()
status.to_csv(HERE/'results/yaapt_notebook_status.csv',index=False)
display(table.query("evaluation=='nested'")[['family','option_id','file','average_mape','F0std_mape','F0num_mape','macro_f1','false_voiced_sil']])
display(status)
print('Mục tiêu từng file đạt:',bool(status.all_files_le_2.any()))


## Các thành phần lỗi

Mỗi cột cộng lỗi mean/std/count chia3; chiều cao là Average MAPE. Đường2% áp dụng mỗi file. Fixed là đánh giá một cấu hình chung, khác nested. Giữ mọi cấu hình thất bại. Không ghép riêng bestphone và beststudio theo chính kết quả held.


In [ ]:
audit.HERE,audit.RESULTS,audit.FIGURES=HERE,HERE/'results',HERE/'figures'
audit.ARTIFACTS.clear()
plot_source=HERE/'results/yaapt_notebook_per_file.csv'
for family in families:
    order=[x['id'] for x in registries[family]]
    labels=['Praat .45']+[f'YAAPT{x["frame_ms"]}' for x in registries[family][1:]]
    fixed=table[(table.evaluation=='fixed')&(table.family==family)]
    fig,axes=audit.plt.subplots(2,2,figsize=(12,8))
    for ax,(file,part) in zip(axes.flat,fixed.groupby('file')):
        part=part.set_index('option_id').loc[order]
        bottom=np.zeros(len(order))
        for metric,label in [('F0mean_mape','mean'),('F0std_mape','std'),('F0num_mape','count')]:
            values=part[metric].to_numpy()/3
            ax.bar(labels,values,bottom=bottom,label=label)
            bottom+=values
        assert np.allclose(bottom,part.average_mape)
        ax.axhline(2,color='black',linestyle='--',label='target 2%')
        ax.set(title=file,ylabel='Average MAPE (%)')
        ax.tick_params(axis='x',rotation=20)
    axes[0,0].legend(fontsize=8)
    fig.suptitle(f'{family}: fixed results recomputed from WAV')
    audit.save_figure(f'yaapt_notebook_{family}_components',fig,[plot_source],
                      'Các thành phần mean/std/count cộng thành Average MAPE; mọi cấu hình fixed.',
                      'Không chọn best theo held file; LAB chưa có F0 chuẩn từng khung.')
    display(fig)
    audit.plt.close(fig)
for figure in audit.ARTIFACTS:
    figure.update(generator='build_yaapt_notebook.py',generator_sha256=audit.digest(HERE/'build_yaapt_notebook.py'),command='python research_workbench_2026_10_07/execute_yaapt_notebook.py')
audit.json_write(HERE/'results/yaapt_notebook_figure_manifest.json',{'figures':audit.ARTIFACTS})
print('Đã xuất PNG/SVG từ các số liệu vừa tính lại; spectral window ghi trên trục x; TDA35ms giữ.')


## Provenance

Lưu input/source/params/fit/calls/contours để đối chiếu độc lập. AMDF thực sự tính lại từ WAV, không sao chép bảng. Thống kê gầnGT không xác minh từng cao độ đúng.


In [ ]:
source_hashes={str(Path(m.__file__).relative_to(REPO)):audit.digest(m.__file__) for m in (dual,reference,core,audit)}
source_hashes.update({key:value for receipt in receipts.values() for key,value in receipt['code_sha256'].items()})
audit.json_write(HERE/'results/yaapt_notebook_provenance.json',{
    'source_sha256':source_hashes,'wav_sha256':{x['file']:audit.digest(core.TRAIN/x['file']) for x in items},
    'native_calls':native_calls,'fit_logs':fit_logs,'praat_native_sha256':native_proof['exe_sha256'],
    'yaapt_port_commit':yaapt_proof['python_port_commit'],'test_read':False,'new_grid_or_selection':False})
assert len(native_calls)==16 and len(fit_logs)==24
print('PASS24 measured rows: H24 AMDF control4 + H40 fixed16 + nested4.')
print('Native/backend16 calls, source/WAV/fit exclusions/contours đã lưu trong provenance; original/frozen giữ.')


## Nguồn và giới hạn

YAAPT_SOURCE_NOTE.md có paperabstract DOI10.1121/1.2916590, authorHTML/manual, pinnedMITsource và workflowcitation K-Dense. Không fullpaper/PDF extraction. Probe rich/sine subharmonic failures giữ; zero/two-harmonic thành công không bảo đảm tiếng nói. H40_REPORT.md/H40_ERROR_ANALYSIS.md tách fixed/nested. Notebook chạy năm cell nguyênsource bằngPythonexec/headlessdisplay, khôngJupyterkernel. Khôngtest/Drive/deep learning/Jev retry; originalnotebooks/frozen giữ. NhãnF/M/device chỉ bốn quan sát, chưa đủ suy rộng.
